# 08 · Backtesting & Accuracy Metrics

Before trusting a forecast, measure it. We **hold out** the last H points,
forecast them, and compare against the truth using MAE, RMSE, MAPE and
**interval coverage** (does the 80% band actually contain 80% of the truth?).

In [ ]:
import torch
import numpy as np
import timesfm

torch.set_float32_matmul_precision("high")

# Downloads ~800 MB of weights the first time, then caches in ~/.cache/huggingface/
model = timesfm.TimesFM_2p5_200M_torch.from_pretrained(
    "google/timesfm-2.5-200m-pytorch"
)

model.compile(
    timesfm.ForecastConfig(
        max_context=1024,
        max_horizon=256,
        normalize_inputs=True,
        use_continuous_quantile_head=True,
        force_flip_invariance=True,
        infer_is_positive=True,
        fix_quantile_crossing=True,
    )
)
print("Model loaded and compiled.")

In [ ]:
rng = np.random.default_rng(21)
t = np.arange(365)
full = (500 + 0.3*t + 60*np.sin(2*np.pi*t/365) + 25*np.sin(2*np.pi*t/7)
        + rng.normal(0, 12, t.size)).astype(np.float32)

H = 30
train, actual = full[:-H], full[-H:]
point, q = model.forecast(horizon=H, inputs=[train])
pred = point[0]

In [ ]:
mae  = np.mean(np.abs(actual - pred))
rmse = np.sqrt(np.mean((actual - pred) ** 2))
mape = np.mean(np.abs((actual - pred) / actual)) * 100
smape = 100 * np.mean(2*np.abs(pred-actual) / (np.abs(pred)+np.abs(actual)))
coverage_80 = np.mean((actual >= q[0, :, 1]) & (actual <= q[0, :, 9])) * 100

print(f"MAE           : {mae:8.2f}")
print(f"RMSE          : {rmse:8.2f}")
print(f"MAPE          : {mape:7.2f}%")
print(f"sMAPE         : {smape:7.2f}%")
print(f"80% coverage  : {coverage_80:7.1f}%   (ideal ~80%)")

## Visual check: forecast vs. held-out truth

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

xf = range(len(train), len(train)+H)
fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(range(len(train))[-90:], train[-90:], color="tab:blue", label="train")
ax.plot(xf, actual, color="black", lw=2, label="actual (held out)")
ax.plot(xf, pred, color="tab:orange", lw=2, ls="--", label="forecast")
ax.fill_between(xf, q[0,:,1], q[0,:,9], color="tab:orange", alpha=0.2, label="80% PI")
ax.legend(); ax.set_title("Backtest: forecast vs. truth")
fig.tight_layout(); fig.savefig("backtest.png", dpi=130)
print("saved backtest.png")

## Rolling-origin backtest (more robust)

A single holdout can be lucky. Evaluate over several cut-off points and average
the errors.

In [ ]:
def rolling_backtest(series, H=30, folds=5, step=20):
    maes = []
    end = len(series)
    for k in range(folds):
        cut = end - k*step
        tr, ac = series[:cut-H], series[cut-H:cut]
        if len(tr) < 64:
            break
        p, _ = model.forecast(horizon=H, inputs=[tr])
        maes.append(np.mean(np.abs(ac - p[0])))
    return np.array(maes)

maes = rolling_backtest(full, H=30, folds=5, step=20)
print("per-fold MAE:", maes.round(2))
print("mean MAE    :", maes.mean().round(2), "+/-", maes.std().round(2))